In [3]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import json
import re
import numpy as np
import pandas as pd


REQUIRED_FILE_KEYS = [
    "field_core_nd.csv",
    "bc_inlet_nd.csv",
    "bc_outlet_nd.csv",
    "bc_wall_nd.csv",
    "bc_body_nd.csv",
]

OPTIONAL_FILE_KEYS = [
    "field_extra_nd.csv",
    "cd_history_clean.csv",
    "cl_history_clean.csv",
    "deltap_history_clean.csv",
    "La_clean.csv",
    "La_value_clean.csv",
    "La_history_clean.csv",
]

FORBIDDEN_LEGACY_FILE_PATTERNS = [
    re.compile(r"(^|[._-])p_front([._-]|$)", re.IGNORECASE),
    re.compile(r"(^|[._-])p_rear([._-]|$)", re.IGNORECASE),
    re.compile(r"(^|[._-])front_pressure([._-]|$)", re.IGNORECASE),
    re.compile(r"(^|[._-])rear_pressure([._-]|$)", re.IGNORECASE),
]

REQ_X_COLS = {"x_nd", "y_nd", "phi_nd", "shape", "Re"}
REQ_Y_COLS = {"u_nd", "v_nd", "p_nd"}

RE_RE_FOLDER = re.compile(r"Re(\d+)", re.IGNORECASE)


def discover_nd_dirs(root: Path) -> List[Path]:
    return sorted({p.parent for p in root.rglob("nd/case_summary.json")})


def _read_json(path: Path) -> Dict:
    return json.loads(path.read_text(encoding="utf-8"))


def _read_cols(csv_path: Path, nrows: int = 5) -> List[str]:
    return list(pd.read_csv(csv_path, nrows=nrows).columns)


def _rows_count(csv_path: Path) -> int:
    with open(csv_path, "rb") as f:
        return max(0, sum(1 for _ in f) - 1)


def _is_number(x) -> bool:
    try:
        if x is None:
            return False
        if isinstance(x, str) and x.strip() == "":
            return False
        return bool(np.isfinite(float(x)))
    except Exception:
        return False


def _folder_Re(nd_dir: Path) -> Optional[int]:
    for parent in nd_dir.parents:
        m = RE_RE_FOLDER.fullmatch(parent.name)
        if m:
            return int(m.group(1))
    return None


def _folder_shape(nd_dir: Path) -> Optional[str]:
    try:
        return nd_dir.parent.parent.name
    except Exception:
        return None


def _resolve_file(files: Dict[str, str], nd_dir: Path, key: str) -> Path:
    raw = files.get(key, None)
    if raw is None:
        return nd_dir / key

    p = Path(str(raw))
    candidates: List[Path] = []

    if p.is_absolute():
        candidates.append(p)
        candidates.append(nd_dir / p.name)
    else:
        candidates.append(nd_dir / p)
        candidates.append(nd_dir / p.name)
        candidates.append(p)

    for c in candidates:
        if c.exists():
            return c

    return candidates[0] if candidates else (nd_dir / key)


def _has_legacy_name(name: str) -> bool:
    return any(p.search(name) for p in FORBIDDEN_LEGACY_FILE_PATTERNS)


def _scan_legacy_pressure_files(nd_dir: Path, files_raw: Dict[str, str]) -> List[str]:
    hits: List[str] = []

    for key, raw in files_raw.items():
        key_s = str(key)
        raw_s = str(raw)
        raw_name = Path(raw_s).name

        if _has_legacy_name(key_s):
            hits.append(f"files_key:{key_s}")
        if _has_legacy_name(raw_s):
            hits.append(f"files_value:{raw_s}")
        if _has_legacy_name(raw_name):
            hits.append(f"files_value_name:{raw_name}")

    if nd_dir.exists():
        for p in nd_dir.iterdir():
            if p.is_file() and _has_legacy_name(p.name):
                hits.append(f"nd_dir_file:{p.name}")

    return sorted(set(hits))


def make_X(df: pd.DataFrame, Re_max: float = 30.0) -> np.ndarray:
    x = df["x_nd"].to_numpy(np.float32)
    y = df["y_nd"].to_numpy(np.float32)
    Re = df["Re"].to_numpy(np.float32) / np.float32(Re_max)
    phi = df["phi_nd"].to_numpy(np.float32)
    return np.stack([x, y, Re, phi], axis=1)


def make_Y(df: pd.DataFrame) -> np.ndarray:
    u = df["u_nd"].to_numpy(np.float32)
    v = df["v_nd"].to_numpy(np.float32)
    p = df["p_nd"].to_numpy(np.float32)
    return np.stack([u, v, p], axis=1)


def _read_last_value(path: Path, value_col: str) -> float:
    try:
        df = pd.read_csv(path)
        if value_col not in df.columns or len(df) == 0:
            return float("nan")
        v = pd.to_numeric(df[value_col], errors="coerce").to_numpy()
        v = v[np.isfinite(v)]
        return float(v[-1]) if len(v) else float("nan")
    except Exception:
        return float("nan")


def _read_La_nd(path: Path) -> float:
    try:
        df = pd.read_csv(path)
        if "La_nd" not in df.columns or len(df) == 0:
            return float("nan")
        s = pd.to_numeric(df["La_nd"], errors="coerce")
        s = s[np.isfinite(s)]
        return float(s.iloc[-1]) if len(s) else float("nan")
    except Exception:
        return float("nan")


def _pick_first_numeric(d: Dict, keys: List[str]) -> float:
    for k in keys:
        if k in d:
            try:
                v = d[k]
                if v is None:
                    continue
                if isinstance(v, str) and v.strip() == "":
                    continue
                fv = float(v)
                if np.isfinite(fv):
                    return fv
            except Exception:
                continue
    return float("nan")


def _point2_to_tuple(value) -> Optional[Tuple[float, float]]:
    if isinstance(value, (list, tuple)) and len(value) == 2:
        try:
            a = float(value[0])
            b = float(value[1])
            if np.isfinite(a) and np.isfinite(b):
                return a, b
        except Exception:
            return None

    if isinstance(value, dict):
        for kx, ky in [("x", "y"), ("x_nd", "y_nd"), ("x_m", "y_m")]:
            if kx in value and ky in value:
                try:
                    a = float(value[kx])
                    b = float(value[ky])
                    if np.isfinite(a) and np.isfinite(b):
                        return a, b
                except Exception:
                    return None
    return None


def _phi_nearest(
    xy_nd: Tuple[float, float],
    field_X: np.ndarray,
    body_X: Optional[np.ndarray] = None,
) -> float:
    xq, yq = float(xy_nd[0]), float(xy_nd[1])

    if body_X is not None and body_X.ndim == 2 and body_X.shape[0] > 0:
        dx = body_X[:, 0] - xq
        dy = body_X[:, 1] - yq
        j = int(np.argmin(dx * dx + dy * dy))
        d2_body = float(dx[j] * dx[j] + dy[j] * dy[j])
        if d2_body <= 1e-8:
            return float(body_X[j, 3])

    dx = field_X[:, 0] - xq
    dy = field_X[:, 1] - yq
    j = int(np.argmin(dx * dx + dy * dy))
    return float(field_X[j, 3])


def _make_probe_X(
    xy_nd: Tuple[float, float],
    Re_case: int,
    Re_max: float,
    field_X: np.ndarray,
    body_X: Optional[np.ndarray],
) -> np.ndarray:
    phi = _phi_nearest(xy_nd, field_X, body_X)
    return np.array(
        [[
            float(xy_nd[0]),
            float(xy_nd[1]),
            float(Re_case) / float(Re_max),
            float(phi),
        ]],
        dtype=np.float32,
    )


@dataclass
class CasePaths:
    nd_dir: Path
    case_id: str
    shape: str
    Re: int
    files: Dict[str, Path]
    scales: Dict[str, float]
    final_values: Dict[str, float]
    has_reference_points: bool
    reference_points: Dict


@dataclass
class CaseData:
    paths: CasePaths
    field_X: np.ndarray
    field_Y: np.ndarray
    has_data: bool
    bc: Dict[str, Dict[str, np.ndarray]]
    targets: Dict[str, float]
    extra: Optional[Dict[str, np.ndarray]]
    probes: Optional[Dict[str, np.ndarray]]


def validate_case_nd(
    nd_dir: Path,
    require_nonempty_bcs: bool = True,
    reject_legacy_pressure_files: bool = True,
) -> Tuple[bool, List[str], Optional[CasePaths]]:
    reasons: List[str] = []

    cs = nd_dir / "case_summary.json"
    if not cs.exists():
        return False, ["missing_file:case_summary.json"], None

    try:
        meta = _read_json(cs)
    except Exception as e:
        return False, [f"bad_json:case_summary.json:{type(e).__name__}"], None

    files_raw = meta.get("files", {})
    if not isinstance(files_raw, dict):
        files_raw = {}

    fshape = _folder_shape(nd_dir)
    fRe = _folder_Re(nd_dir)

    case_id = str(meta.get("case_id", nd_dir.parent.name))
    shape = str(meta.get("shape", fshape if fshape is not None else ""))

    Re_raw = meta.get("Re", fRe if fRe is not None else -1)
    try:
        Re = int(float(Re_raw))
    except Exception:
        Re = fRe if fRe is not None else -1

    if fshape and shape and fshape.lower() != shape.lower():
        reasons.append(f"shape_mismatch:folder={fshape},summary={shape}")
    if fRe is not None and Re != -1 and fRe != Re:
        reasons.append(f"Re_mismatch:folder={fRe},summary={Re}")

    for top_key in ["scales", "files", "final_values", "notes"]:
        if top_key not in meta:
            reasons.append(f"missing_case_summary_key:{top_key}")

    if ("body_info" not in meta) and ("polygon_vertices_m" not in meta):
        reasons.append("missing_geometry_info:need_body_info_or_polygon_vertices_m")

    if reject_legacy_pressure_files:
        legacy_hits = _scan_legacy_pressure_files(nd_dir, files_raw)
        for hit in legacy_hits:
            reasons.append(f"legacy_pressure_file:{hit}")

    resolved: Dict[str, Path] = {}
    for key in REQUIRED_FILE_KEYS:
        p = _resolve_file(files_raw, nd_dir, key)
        resolved[key] = p
        if not p.exists():
            reasons.append(f"missing_file:{key}")

    if any(r.startswith("missing_file:") for r in reasons):
        return False, reasons, None

    p_field = resolved["field_core_nd.csv"]
    cols_field = set(_read_cols(p_field))
    if not REQ_X_COLS.issubset(cols_field):
        missing = sorted(list(REQ_X_COLS - cols_field))
        reasons.append(f"field_core_missing_X_cols:{missing}")

    for key in ["bc_inlet_nd.csv", "bc_outlet_nd.csv", "bc_wall_nd.csv", "bc_body_nd.csv"]:
        cols_bc = set(_read_cols(resolved[key]))
        if not REQ_X_COLS.issubset(cols_bc):
            missing = sorted(list(REQ_X_COLS - cols_bc))
            reasons.append(f"{key}_missing_X_cols:{missing}")

    if _rows_count(p_field) <= 0:
        reasons.append("empty_csv:field_core_nd.csv")

    if require_nonempty_bcs:
        for key in ["bc_inlet_nd.csv", "bc_outlet_nd.csv", "bc_wall_nd.csv", "bc_body_nd.csv"]:
            if _rows_count(resolved[key]) <= 0:
                reasons.append(f"empty_csv:{key}")

    for key in OPTIONAL_FILE_KEYS:
        p = _resolve_file(files_raw, nd_dir, key)
        if p.exists():
            resolved[key] = p

    scales_raw = meta.get("scales", {})
    if not isinstance(scales_raw, dict):
        scales_raw = {}

    final_values_raw = meta.get("final_values", {})
    if not isinstance(final_values_raw, dict):
        final_values_raw = {}

    scales = {k: float(v) for k, v in scales_raw.items() if _is_number(v)}
    final_values = {k: float(v) for k, v in final_values_raw.items() if _is_number(v)}
    reference_points = meta.get("reference_points", {})
    if not isinstance(reference_points, dict):
        reference_points = {}

    if reasons:
        return False, reasons, None

    return True, ["OK"], CasePaths(
        nd_dir=nd_dir,
        case_id=case_id,
        shape=shape,
        Re=Re,
        files=resolved,
        scales=scales,
        final_values=final_values,
        has_reference_points=bool(reference_points),
        reference_points=reference_points,
    )


def load_case(paths: CasePaths, Re_max: float = 30.0) -> CaseData:
    p_field = paths.files["field_core_nd.csv"]
    cols = set(_read_cols(p_field))

    usecols = ["x_nd", "y_nd", "phi_nd", "shape", "Re"]
    has_data = REQ_Y_COLS.issubset(cols)

    if has_data:
        df_f = pd.read_csv(p_field, usecols=usecols + ["u_nd", "v_nd", "p_nd"])
        Xf = make_X(df_f, Re_max=Re_max)
        Yf = make_Y(df_f)
    else:
        df_f = pd.read_csv(p_field, usecols=usecols)
        Xf = make_X(df_f, Re_max=Re_max)
        Yf = np.zeros((len(df_f), 3), dtype=np.float32)

    bc: Dict[str, Dict[str, np.ndarray]] = {}
    for bc_type, key in [
        ("inlet", "bc_inlet_nd.csv"),
        ("outlet", "bc_outlet_nd.csv"),
        ("wall", "bc_wall_nd.csv"),
        ("body", "bc_body_nd.csv"),
    ]:
        p = paths.files[key]
        cols_bc = set(_read_cols(p))

        usecols_bc = ["x_nd", "y_nd", "phi_nd", "shape", "Re"]
        extra_cols = [c for c in ["u_nd", "v_nd", "p_nd"] if c in cols_bc]
        df_b = pd.read_csv(p, usecols=usecols_bc + extra_cols)

        Xb = make_X(df_b, Re_max=Re_max)
        has_Y = all(c in df_b.columns for c in ["u_nd", "v_nd", "p_nd"])
        Yb = make_Y(df_b) if has_Y else np.zeros((len(df_b), 3), dtype=np.float32)

        bc[bc_type] = {
            "X": Xb,
            "Y": Yb,
            "has_Y": bool(has_Y),
        }

    targets: Dict[str, float] = {
        "Cd": float("nan"),
        "Cl": float("nan"),
        "deltap_nd": float("nan"),
        "La_nd": float("nan"),
    }

    fv = paths.final_values

    targets["Cd"] = _pick_first_numeric(fv, ["Cd_final_used", "Cd_mean", "Cd_last", "Cd"])
    targets["Cl"] = _pick_first_numeric(fv, ["Cl_final_used", "Cl_mean", "Cl_last", "Cl"])
    targets["deltap_nd"] = _pick_first_numeric(
        fv,
        ["deltap_nd_final_used", "deltap_nd_mean", "deltap_nd_last", "deltap_nd"],
    )
    targets["La_nd"] = _pick_first_numeric(
        fv,
        ["La_nd_final_used", "La_nd_mean", "La_nd"],
    )

    if np.isnan(targets["Cd"]) and "cd_history_clean.csv" in paths.files:
        targets["Cd"] = _read_last_value(paths.files["cd_history_clean.csv"], "Cd")

    if np.isnan(targets["Cl"]) and "cl_history_clean.csv" in paths.files:
        targets["Cl"] = _read_last_value(paths.files["cl_history_clean.csv"], "Cl")

    if np.isnan(targets["deltap_nd"]) and "deltap_history_clean.csv" in paths.files:
        targets["deltap_nd"] = _read_last_value(paths.files["deltap_history_clean.csv"], "deltap_nd")

    if np.isnan(targets["La_nd"]) and "La_value_clean.csv" in paths.files:
        targets["La_nd"] = _read_La_nd(paths.files["La_value_clean.csv"])

    if np.isnan(targets["La_nd"]) and "La_history_clean.csv" in paths.files:
        targets["La_nd"] = _read_last_value(paths.files["La_history_clean.csv"], "La_nd")

    extra_out: Optional[Dict[str, np.ndarray]] = None
    p_extra = paths.files.get("field_extra_nd.csv", None)
    if p_extra is not None and p_extra.exists():
        cols_extra = set(_read_cols(p_extra))
        needed = {"x_nd", "y_nd", "phi_nd", "shape", "Re", "dpdx_nd", "dpdy_nd"}
        if needed.issubset(cols_extra):
            df_e = pd.read_csv(
                p_extra,
                usecols=["x_nd", "y_nd", "phi_nd", "shape", "Re", "dpdx_nd", "dpdy_nd"],
            )
            extra_out = {
                "X": make_X(df_e, Re_max=Re_max),
                "dpdx": df_e["dpdx_nd"].to_numpy(np.float32).reshape(-1, 1),
                "dpdy": df_e["dpdy_nd"].to_numpy(np.float32).reshape(-1, 1),
            }

    probes_out: Optional[Dict[str, np.ndarray]] = None
    front_nd = _point2_to_tuple(paths.reference_points.get("deltap_front_nd"))
    rear_nd = _point2_to_tuple(paths.reference_points.get("deltap_rear_nd"))
    if front_nd is not None and rear_nd is not None:
        body_X = bc["body"]["X"] if "body" in bc else None
        probes_out = {
            "front_X": _make_probe_X(front_nd, paths.Re, Re_max, Xf, body_X),
            "rear_X": _make_probe_X(rear_nd, paths.Re, Re_max, Xf, body_X),
        }

    return CaseData(
        paths=paths,
        field_X=Xf,
        field_Y=Yf,
        has_data=bool(has_data),
        bc=bc,
        targets=targets,
        extra=extra_out,
        probes=probes_out,
    )


def load_all_cases(
    root: Path,
    Re_max: float = 30.0,
    verbose: bool = True,
) -> List[CaseData]:
    nd_dirs = discover_nd_dirs(root)

    if len(nd_dirs) == 0:
        if verbose:
            print("\n=== LOADER SUMMARY ===")
            print(f"root: {root}")
            print("discovered_nd_dirs: 0")
            print("loaded_cases: 0")
        return []

    cases: List[CaseData] = []
    skipped: List[Tuple[Path, List[str]]] = []

    for nd in nd_dirs:
        ok, reasons, paths = validate_case_nd(nd)
        if not ok:
            skipped.append((nd, reasons))
            continue

        assert paths is not None
        cases.append(load_case(paths, Re_max=Re_max))

    if verbose:
        print("\n=== LOADER SUMMARY ===")
        print(f"root: {root}")
        print(f"discovered_nd_dirs: {len(nd_dirs)}")
        print(f"loaded_cases: {len(cases)}")
        print(f"skipped_cases: {len(skipped)}")

        print("\n=== CASES USED BY LOADER ===")
        for c in cases:
            has_extra = c.extra is not None and c.extra["X"].shape[0] > 0
            has_probes = c.probes is not None
            print(
                f"- {c.paths.shape} | Re={c.paths.Re} | case_id={c.paths.case_id} | "
                f"has_data={c.has_data} | has_extra={has_extra} | has_probes={has_probes} | "
                f"targets(Cd,Cl,dp,La)=({c.targets.get('Cd', np.nan):.3g},"
                f"{c.targets.get('Cl', np.nan):.3g},"
                f"{c.targets.get('deltap_nd', np.nan):.3g},"
                f"{c.targets.get('La_nd', np.nan):.3g})"
            )

    return cases


if __name__ == "__main__":
    ROOT = Path(r"D:\Projects\Simulations\AI FInal\Ansys\Shapes")
    cases = load_all_cases(ROOT, Re_max=30.0, verbose=True)


=== LOADER SUMMARY ===
root: D:\Projects\Simulations\AI FInal\Ansys\Shapes
discovered_nd_dirs: 9
loaded_cases: 9
skipped_cases: 0

=== CASES USED BY LOADER ===
- Circle | Re=10 | case_id=Circle_2D_Re10 | has_data=True | has_extra=True | has_probes=True | targets(Cd,Cl,dp,La)=(8.09,0.0682,3.45,0.176)
- Circle | Re=20 | case_id=Circle_2D_Re20 | has_data=True | has_extra=True | has_probes=True | targets(Cd,Cl,dp,La)=(5.29,0.00726,2.65,0.762)
- Circle | Re=30 | case_id=Circle_2D_Re30 | has_data=True | has_extra=True | has_probes=True | targets(Cd,Cl,dp,La)=(4.3,-0.00423,2.4,1.26)
- Hexagon | Re=10 | case_id=Hexagon_2D_Re10 | has_data=True | has_extra=True | has_probes=True | targets(Cd,Cl,dp,La)=(8.86,0.0864,3.71,0.198)
- Hexagon | Re=20 | case_id=Hexagon_2D_Re20 | has_data=True | has_extra=True | has_probes=True | targets(Cd,Cl,dp,La)=(5.8,0.0156,2.82,0.824)
- Hexagon | Re=30 | case_id=Hexagon_2D_Re30 | has_data=True | has_extra=True | has_probes=True | targets(Cd,Cl,dp,La)=(4.73,0.0025,